# Day 2 — Stateful Workflow Orchestration: Graphs, Routes, and Observability

**Course:** Agentic Systems for Forward Deployed Engineers  
**Theme:** A validated invoice isn't processed — it's *routed*. Day 2 turns the `CanonicalInvoice` from Day 1 into a living workflow with observable state, conditional logic, and full cost tracing.

---

## What this day covers

| Layer | Skill |
|---|---|
| State design | A single `WorkflowState` object carries all workflow context — no global vars |
| Conditional routing | `decide_route` applies business rules; LangGraph wires the conditional edges |
| Idempotency | `ensure_action_once` prevents duplicate side-effects on retry |
| Observability | `traced_node` decorator captures latency/cost per node automatically |
| Graph compilation | `build_graph()` assembles a LangGraph `StateGraph` from nodes and edges |

## Why this matters for FDEs

Production workflows break in two predictable ways: they re-execute side-effects on retry, and they lose the audit trail of how a decision was made.  
Day 2 solves both with idempotency keys and an append-only evidence log carried inside the state object.

---

## Prerequisites

```bash
pip install -r requirements-day2.txt
```

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print("✓ repo root:", repo_root)

---
## Part 1 — The Workflow State

All information a workflow needs — the invoice, routing decision, evidence trail, cost metrics, recommendations — lives in `WorkflowState`.  
There is no global state. No class-level variables. Each workflow run is fully self-contained.

### Why a single state object?

LangGraph threads state through every node automatically.  
Any node can read the whole state, any node can mutate it.  
This makes the flow inspectable: you can snapshot state at any point and know exactly what has happened.

In [ ]:
from aegisap.day2.state import WorkflowState, make_initial_state
from aegisap.day2.config import HIGH_VALUE_THRESHOLD, ROUTE_PRECEDENCE, KNOWN_VENDORS
from aegisap.day_01.fixture_loader import load_fixture_package, load_fixture_candidate
from aegisap.day_01.service import canonicalize_with_candidate
import json

# Bootstrap from a fixture — same canonicalization as Day 1
case = "happy_path"
pkg = load_fixture_package(case)
cand = load_fixture_candidate(case)
invoice = canonicalize_with_candidate(pkg, cand)

state = make_initial_state(
    invoice,
    package_id=pkg.message_id,
    known_vendor=invoice.supplier_name in KNOWN_VENDORS,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

print(f"workflow_id:          {state.workflow_id}")
print(f"status:               {state.status}")
print(f"current_node:         {state.current_node}")
print(f"invoice:              {state.invoice.invoice_number} | {state.invoice.currency} {state.invoice.gross_amount}")
print(f"vendor.is_known:      {state.vendor.is_known_vendor}")
print(f"policy.threshold:     {state.policy.high_value_threshold}")
print(f"evidence count:       {len(state.evidence)}")
print(f"recommendations:      {len(state.recommendations)}")
print(f"completed_nodes:      {state.completed_nodes}")

---
## Part 2 — Routing Logic

The router applies three predicates in precedence order:

```
high_value  →  invoice.gross_amount >= threshold
new_vendor  →  NOT vendor.is_known_vendor
clean_path  →  all other invoices
```

**Precedence** means that if an invoice is both high-value *and* from a new vendor, it routes to `high_value` — not both.  
The `ROUTE_PRECEDENCE` list in `config.py` controls this ordering.

In [ ]:
from aegisap.day2.predicates import is_high_value, is_new_vendor
from aegisap.day2.router import decide_route, RouteDecision
from aegisap.day2.config import HIGH_VALUE_THRESHOLD, ROUTE_PRECEDENCE
from decimal import Decimal

# Test routing for different scenarios
scenarios = [
    {"case": "clean_path",   "known_vendor": True,  "gross_amount": Decimal("500.00")},
    {"case": "high_value",   "known_vendor": True,  "gross_amount": Decimal("15000.00")},
    {"case": "new_vendor",   "known_vendor": False, "gross_amount": Decimal("500.00")},
    {"case": "both_flags",   "known_vendor": False, "gross_amount": Decimal("15000.00")},
]

print(f"{'Scenario':<20} {'is_hv':<8} {'is_nv':<8} {'route':<15} {'reason'}")
print("-" * 80)

for sc in scenarios:
    # Build a state with the needed properties
    from copy import deepcopy
    test_state = make_initial_state(
        invoice,
        package_id="test",
        known_vendor=sc["known_vendor"],
        high_value_threshold=HIGH_VALUE_THRESHOLD,
        route_precedence=ROUTE_PRECEDENCE,
    )
    # Override gross_amount via model copy (frozen invoice — build a new one)
    from aegisap.day_01.models import CanonicalInvoice
    d = invoice.model_dump(mode="json")
    gross = sc["gross_amount"]
    # Keep net+vat+gross consistent (no VAT, simple)
    d["gross_amount"] = str(gross)
    d["net_amount"] = str(gross)
    d["vat_amount"] = "0.00"
    new_invoice = CanonicalInvoice.model_validate(d)
    test_state.invoice = new_invoice

    hv = is_high_value(test_state)
    nv = is_new_vendor(test_state)
    decision = decide_route(test_state)
    print(f"{sc['case']:<20} {str(hv):<8} {str(nv):<8} {decision.route:<15} {decision.reason[:40]}")

---
## Part 3 — LangGraph: Building the Workflow Graph

LangGraph compiles a `StateGraph` into an executable application.  
The graph is declarative: you define nodes and edges, then call `.compile()`.  
At runtime, LangGraph threads `WorkflowState` through each node in order.

```
START
  └─ init_workflow
       └─ route_invoice
            ├─ [high_value]  → high_value_review  ┐
            ├─ [new_vendor]  → new_vendor_review   ├─ finalize_workflow → END
            └─ [clean_path]  → clean_path_finalize ┘
```

In [ ]:
try:
    from aegisap.day2.graph import build_graph
    app = build_graph()
    print("✓ Graph compiled successfully")
    print("  Nodes:", list(app.nodes.keys()) if hasattr(app, 'nodes') else "(use app.get_graph() to inspect)")
except ImportError as e:
    print(f"⚠ LangGraph not installed: {e}")
    print("  Run: pip install -r requirements-day2.txt")

---
## Part 4 — Nodes: What Each Step Does

Each node is a plain Python function that receives `WorkflowState` and returns `WorkflowState`.  
The `@traced_node` decorator wraps each node to capture timing automatically.  
Nodes are composable and testable in isolation — no mocking required.

In [ ]:
from aegisap.day2.nodes import init_workflow, route_invoice, high_value_review, clean_path_finalize, finalize_workflow
from aegisap.day_01.fixture_loader import load_fixture_package, load_fixture_candidate
from aegisap.day_01.service import canonicalize_with_candidate
from aegisap.day2.config import HIGH_VALUE_THRESHOLD, ROUTE_PRECEDENCE

pkg = load_fixture_package("happy_path")
cand = load_fixture_candidate("happy_path")
invoice = canonicalize_with_candidate(pkg, cand)

state = make_initial_state(
    invoice,
    package_id=pkg.message_id,
    known_vendor=True,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

# Walk nodes manually to observe each step
print("=== init_workflow ===")
state = init_workflow(state)
print(f"  status: {state.status}")
print(f"  evidence items: {len(state.evidence)}")
print(f"  completed_nodes: {state.completed_nodes}")

print("\n=== route_invoice ===")
state = route_invoice(state)
print(f"  route: {state.route}")
print(f"  route_reason: {state.route_reason}")
print(f"  evidence items: {len(state.evidence)}")

print("\n=== clean_path_finalize ===")
state = clean_path_finalize(state)
print(f"  status: {state.status}")

print("\n=== finalize_workflow ===")
state = finalize_workflow(state)
print(f"  total_latency_ms: {state.total_latency_ms}")
print(f"  node_metrics count: {len(state.node_metrics)}")

---
## Part 5 — Idempotency

In distributed systems, a node may be re-executed due to retry, network blip, or checkpoint replay.  
Without idempotency guards, every re-execution would fire a duplicate notification, approval request, or payment.  

`ensure_action_once` generates a deterministic key from `(workflow_id, node_name, action)` and stores it in the state.  
If the same action is attempted again in the same workflow, it returns `False` — and the caller skips the side-effect.

In [ ]:
from aegisap.day2.idempotency import ensure_action_once, semantic_idempotency_key

# Simulate a state object
fresh_state = make_initial_state(
    invoice,
    package_id="idem-test",
    known_vendor=False,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

action = "send_approval_request"

print("First call (should fire):")
result = ensure_action_once(fresh_state, "high_value_review", action)
print(f"  ensure_action_once → {result}  (True = proceed, False = skip)")

print("\nSecond call (should be blocked — idempotency key already stored):")
result = ensure_action_once(fresh_state, "high_value_review", action)
print(f"  ensure_action_once → {result}")

print("\nDifferent action on same node (should fire):")
result = ensure_action_once(fresh_state, "high_value_review", "log_threshold_breach")
print(f"  ensure_action_once → {result}")

print("\nIdempotency keys in state:")
for k, v in fresh_state.idempotency_keys.items():
    print(f"  {k}: {v[:16]}...")

---
## Part 6 — Observability: The `traced_node` Decorator

Every node decorated with `@traced_node(name)` automatically appends a `NodeMetric` to the state after execution.  
This gives you per-node latency, token usage, and cost — without any instrumentation code inside the nodes themselves.

In [ ]:
from aegisap.day2.nodes import (
    init_workflow, route_invoice, new_vendor_review, finalize_workflow
)

state = make_initial_state(
    invoice,
    package_id="trace-test",
    known_vendor=False,  # triggers new_vendor route
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

state = init_workflow(state)
state = route_invoice(state)
state = new_vendor_review(state)
state = finalize_workflow(state)

print("Per-node metrics:")
print(f"{'node_name':<25} {'latency_ms':>12} {'outcome'}")
print("-" * 50)
for m in state.node_metrics:
    print(f"{m.node_name:<25} {m.latency_ms:>12} {m.outcome}")

print(f"\nAggregate totals:")
print(f"  total_latency_ms:   {state.total_latency_ms}")
print(f"  total_cost_usd:     {state.total_cost_usd:.6f}")
print(f"  recommendations:    {len(state.recommendations)}")

---
## Part 7 — Running the Full Graph End-to-End

Now we run all three fixture paths through the compiled LangGraph application.  
Observe how the state differs after each run — same graph, different routes.

In [ ]:
try:
    from aegisap.day2.graph import build_graph
    from aegisap.day2.run_workflow import run_from_fixture

    app = build_graph()

    fixture_base = repo_root / "fixtures" / "day2"
    cases = ["clean_path", "high_value", "missing_po", "new_vendor"]

    print(f"{'Fixture':<20} {'route':<15} {'status':<15} {'recommendations'}")
    print("-" * 75)

    for case in cases:
        fixture_dir = fixture_base / case
        if not fixture_dir.exists():
            print(f"{case:<20} (fixture directory not found, skipping)")
            continue
        result = run_from_fixture(fixture_dir)
        rec_names = [r.action for r in result.recommendations]
        print(f"{case:<20} {str(result.route):<15} {result.status:<15} {rec_names}")

except ImportError:
    print("⚠ LangGraph not installed — run: pip install -r requirements-day2.txt")

---
## Part 8 — The Evidence Trail

Every decision a node makes is captured in `state.evidence` as an append-only `EvidenceItem`.  
This is the audit trail that compliance, fraud detection, and post-mortems rely on.  
It is never removed — only appended to.

In [ ]:
state = make_initial_state(
    invoice,
    package_id="evidence-demo",
    known_vendor=False,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

from aegisap.day2.nodes import init_workflow, route_invoice, new_vendor_review, finalize_workflow
state = init_workflow(state)
state = route_invoice(state)
state = new_vendor_review(state)
state = finalize_workflow(state)

print(f"Evidence trail ({len(state.evidence)} items):\n")
for i, ev in enumerate(state.evidence, 1):
    print(f"  [{i}] kind={ev.kind:<30} source={ev.source:<25} detail={ev.detail[:50]}")

---
## Exercises

### Exercise 1 — Extend the routing policy

Add a new route: `locale_mismatch`.  
An invoice is a locale mismatch when its currency does not match the supplier's expected currency.  
For simplicity, assume `"Contoso Ltd"` always invoices in `GBP`.

1. Add `is_locale_mismatch(state: WorkflowState) -> bool` to `predicates.py`.
2. Wire it into `decide_route` with a precedence between `new_vendor` and `clean_path`.
3. Test it in the cell below.

In [ ]:
# Exercise 1 workspace
# Define is_locale_mismatch without modifying the source — extend in-place here
from aegisap.day2.state import WorkflowState

EXPECTED_CURRENCY = {"Contoso Ltd": "GBP", "Rhein Energie GmbH": "EUR"}

def is_locale_mismatch(state: WorkflowState) -> bool:
    expected = EXPECTED_CURRENCY.get(state.invoice.supplier_name)
    if expected is None:
        return False  # unknown vendor — can't assert mismatch
    return state.invoice.currency != expected

# Test it
from copy import deepcopy
from aegisap.day_01.models import CanonicalInvoice

# Build a EUR invoice from Contoso (who normally invoices in GBP)
d = invoice.model_dump(mode="json")
d["currency"] = "EUR"
mismatch_invoice = CanonicalInvoice.model_validate(d)

test_state = make_initial_state(
    mismatch_invoice,
    package_id="ex1",
    known_vendor=True,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

print(f"is_locale_mismatch: {is_locale_mismatch(test_state)}")
print("Expected: True (Contoso invoiced in EUR, not GBP)")

### Exercise 2 — Observe idempotency under simulated retry

Simulate a scenario where `high_value_review` is accidentally called twice (e.g., network retry).  
1. Run `high_value_review` twice on the same state.
2. Verify that only **one** `manager_approval_required` recommendation exists.
3. Count how many `NodeMetric` entries exist — why are there two?

In [ ]:
# Exercise 2 workspace
from aegisap.day_01.models import CanonicalInvoice
from aegisap.day2.nodes import init_workflow, route_invoice, high_value_review
from aegisap.day2.config import HIGH_VALUE_THRESHOLD, ROUTE_PRECEDENCE

# Build a high-value invoice
d = invoice.model_dump(mode="json")
d["gross_amount"] = "50000.00"
d["net_amount"] = "50000.00"
d["vat_amount"] = "0.00"
hv_invoice = CanonicalInvoice.model_validate(d)

state = make_initial_state(
    hv_invoice,
    package_id="retry-test",
    known_vendor=True,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)

state = init_workflow(state)
state = route_invoice(state)

# Simulate double execution
state = high_value_review(state)  # first call
state = high_value_review(state)  # simulated retry

print(f"recommendations count: {len(state.recommendations)}  (expected: 1)")
print(f"node_metrics count:    {len(state.node_metrics)}      (expected: 4 — including 2x high_value_review)")
for r in state.recommendations:
    print(f"  action: {r.action}")

### Exercise 3 — Serialise and deserialise workflow state

In production, workflow state must survive process restarts and be stored in a database.  
1. Serialise a completed `WorkflowState` to JSON.
2. Round-trip it back to a `WorkflowState` using `model_validate`.
3. Confirm the route, status, and evidence count are identical before and after.

In [ ]:
# Exercise 3 workspace
import json
from aegisap.day2.nodes import init_workflow, route_invoice, clean_path_finalize, finalize_workflow

state = make_initial_state(
    invoice,
    package_id="serde-test",
    known_vendor=True,
    high_value_threshold=HIGH_VALUE_THRESHOLD,
    route_precedence=ROUTE_PRECEDENCE,
)
state = init_workflow(state)
state = route_invoice(state)
state = clean_path_finalize(state)
state = finalize_workflow(state)

# Serialise
raw_json = state.model_dump_json(indent=2)

# Deserialise
restored = WorkflowState.model_validate_json(raw_json)

# Verify round-trip
assert restored.route == state.route, "route mismatch!"
assert restored.status == state.status, "status mismatch!"
assert len(restored.evidence) == len(state.evidence), "evidence count mismatch!"

print("✓ Round-trip successful")
print(f"  route:          {restored.route}")
print(f"  status:         {restored.status}")
print(f"  evidence items: {len(restored.evidence)}")
print(f"  JSON size:      {len(raw_json):,} bytes")

---
## Summary — Day 2 Principles

| Principle | Implementation |
|---|---|
| Single state object | `WorkflowState` carries all workflow context — no globals, no hidden mutation |
| Deterministic routing | `decide_route` uses predicates + precedence list — no LLM in the hot path |
| Idempotent side-effects | `ensure_action_once` prevents duplicate actions on retry |
| Automatic observability | `@traced_node` captures per-node metrics without instrumenting node logic |
| Append-only evidence | `state.evidence` is the immutable audit trail for every routing decision |

## What Day 3 builds on this

Day 2 routing is deterministic — it fires rules against the invoice data.  
Day 3 adds *retrieval*: specialist agents pull vendor history, PO records, and policy documents from external stores, each with a trust tier.  
The question becomes: **which source do you trust when they disagree?**